In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

csv_path = Path.cwd() / "datasets" / "Housing.csv"
df = pd.read_csv(csv_path)

In [2]:
binary_cols = ['mainroad', 'guestroom', 'basement', 'prefarea', 'hotwaterheating', 'airconditioning']
df[binary_cols] = df[binary_cols].replace({'yes': 1, 'no': 0})
df['furnishingstatus'] = df['furnishingstatus'].replace(
    {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
)

C:\Users\welcome\AppData\Local\Temp\ipykernel_40132\2722182710.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_cols] = df[binary_cols].replace({'yes': 1, 'no': 0})
C:\Users\welcome\AppData\Local\Temp\ipykernel_40132\2722182710.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['furnishingstatus'] = df['furnishingstatus'].replace(


In [3]:
X = df[['area', 'bedrooms', 'bathrooms', 'stories', 'parking',
        'mainroad', 'guestroom', 'basement', 'prefarea',
        'hotwaterheating', 'airconditioning']]
y = df['price']

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

print(X.shape)
print(X_poly.shape)

(545, 11)
(545, 77)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
results = []

for alpha in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results.append({
        "alpha": alpha,
        "train_r2": model.score(X_train, y_train),
        "test_r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
    })

In [6]:
results_df = pd.DataFrame(results).sort_values("test_r2", ascending=False).reset_index(drop=True)
results_df

,alpha,train_r2,test_r2,mae,rmse
0,100.000,0.732088,0.665404,9.679114e+05,1.300476e+06
1,10.000,0.762206,0.654150,1.007736e+06,1.322166e+06
2,1000.000,0.686502,0.637197,9.835555e+05,1.354183e+06
3,1.000,0.771072,0.636154,1.034797e+06,1.356129e+06
4,0.100,0.771606,0.629207,1.043847e+06,1.369014e+06
5,0.010,0.771615,0.628239,1.045001e+06,1.370800e+06
6,0.001,0.771615,0.628138,1.045120e+06,1.370986e+06


In [7]:
best_alpha = results_df.iloc[0]["alpha"]
print(f"Best alpha by test R2: {best_alpha}")

Best alpha by test R2: 100.0
